In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ (her notebook'un ilk hücresi) ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"   # private repo ise token kullanılır
BRANCH   = "main"                                           # kendi branch'inizde çalışıyorsanız değiştirin
TOKEN    = userdata.get("GH_TOKEN")                         # Colab > Secrets > GH_TOKEN (sadece private repo)
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
if "/content/repo/src" not in sys.path:      # editable install is only seen after a kernel restart
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()          # config/config.yaml + DATA_ROOT
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 00 — Setup: load raw data and build the local sample

Stage 1.0. This notebook contains no business logic; it only calls `recsys`.

1. Reads `DATA_ROOT/raw/RAW_recipes.csv` and `RAW_interactions.csv` with `recsys.data.load`
   (lists parsed, nutrition split into 7 columns, dates parsed, schemas validated).
2. Saves them as stage **`loaded`** (`DATA_ROOT/outputs/loaded/vN/` + `LATEST`, with manifest).
3. Draws a 5,000-recipe sample stratified on `n_ingredients` plus **all** interactions of those
   recipes and saves it as stage **`sample`**. This is copied to the repo as `data/sample/`.

Loading is faithful: nothing is dropped, imputed or clipped here (that is 1.2–1.4).

In [ ]:
from recsys.contracts import LOADED_INTERACTIONS_SCHEMA, LOADED_RECIPES_SCHEMA
from recsys.data.load import (
    load_interactions,
    load_recipes,
    make_stratified_sample,
    stratification_table,
)
from recsys.io import ArtifactStore, describe_file

store = ArtifactStore.from_config(CFG)
NA_VALUES = CFG.section("load")["na_values"]
RAW_RECIPES = CFG.paths["raw_recipes"]
RAW_INTERACTIONS = CFG.paths["raw_interactions"]
print("outputs root:", store.root)

## 1. Load and validate the raw CSVs

In [ ]:
recipes = load_recipes(RAW_RECIPES, NA_VALUES)
interactions = load_interactions(RAW_INTERACTIONS, NA_VALUES)
print("recipes:", recipes.shape, "| interactions:", interactions.shape)
recipes.head(3)

In [ ]:
print(recipes.dtypes.to_string())
print()
print(interactions.dtypes.to_string())

## 2. Save stage `loaded`

In [ ]:
loaded = store.save(
    "loaded",
    {"recipes": recipes, "interactions": interactions},
    inputs=[describe_file(RAW_RECIPES), describe_file(RAW_INTERACTIONS)],
    params={"na_values": NA_VALUES},
    schemas={"recipes": LOADED_RECIPES_SCHEMA, "interactions": LOADED_INTERACTIONS_SCHEMA},
)
print("saved:", loaded.stage, loaded.version, "->", loaded.path)
{name: (f["n_rows"], f["n_cols"]) for name, f in loaded.manifest["files"].items()}

## 3. Stratified sample → stage `sample`

In [ ]:
SAMPLE = CFG.section("sample")
sample_recipes, sample_interactions = make_stratified_sample(
    recipes,
    interactions,
    n_recipes=SAMPLE["n_recipes"],
    stratify_col=SAMPLE["stratify_col"],
    random_state=CFG.random_state,
)
strat = stratification_table(recipes, sample_recipes, SAMPLE["stratify_col"])
sample = store.save(
    "sample",
    {
        "recipes": sample_recipes,
        "interactions": sample_interactions,
        "stratification": strat,
    },
    inputs=[loaded],
    params={**SAMPLE, "random_state": CFG.random_state},
    schemas={"recipes": LOADED_RECIPES_SCHEMA, "interactions": LOADED_INTERACTIONS_SCHEMA},
)
print("saved:", sample.stage, sample.version, "->", sample.path)
print("sample recipes:", sample_recipes.shape, "| sample interactions:", sample_interactions.shape)
print("max |share difference| per stratum:", strat["share_diff"].abs().max())

## 4. Re-read both stages (sha256 verified against the manifests)

In [ ]:
for stage in ("loaded", "sample"):
    art = store.load(stage)
    print(stage, art.version, {k: getattr(v, "shape", None) for k, v in art.objects.items()})

## 5. Copy the sample into the repo (`data/sample/`)

The sample folder is small and is committed so both team members can run local tests.
On a computer with **Google Drive for desktop** syncing `CME4403/data`, from the repo root:

```bash
DATA_ROOT="<synced path>/CME4403/data" python -c "from recsys.config import load_config; from recsys.io import ArtifactStore; ArtifactStore.from_config(load_config()).export('sample', 'data/sample')"
```

`export` verifies every sha256 and refuses to overwrite an existing `data/sample/manifest.json`.
Then `pytest -q` also runs `tests/test_sample.py` on the real sample.